# 02 — Retrieval e RAG fundamentado

Este notebook valida a recuperação antes de conectar uma LLM. Separar retrieval de geração permite descobrir se uma resposta ruim nasceu na busca ou no modelo.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from langchain_core.prompts import ChatPromptTemplate

from ragnaldo.config import SETTINGS
from ragnaldo.ingestion import load_verified_index

vector_store, manifest = load_verified_index()
manifest

## 1. Busca por similaridade

Começamos observando conteúdo e metadados. Uma resposta só poderá citar uma página se a recuperação preservá-la corretamente.

In [ ]:
question = 'O que o ONE AI for Tech ensina sobre RAG e LangChain?'
results = vector_store.similarity_search_with_score(question, k=SETTINGS.retrieval_k)

for rank, (document, score) in enumerate(results, start=1):
    page = document.metadata.get('page')
    page_label = page + 1 if isinstance(page, int) else 'n/a'
    print(f'[{rank}] score={score:.4f} | {document.metadata.get("source")} | página={page_label}')
    print(document.page_content[:400], '\n')

## 2. Contexto com citações

O modelo receberá cada trecho com fonte e página. A citação exibida ao usuário será construída a partir dos mesmos metadados, sem pedir que a LLM invente referências.

In [ ]:
def format_context(documents):
    blocks = []
    for document in documents:
        page = document.metadata.get('page')
        page_label = page + 1 if isinstance(page, int) else 'não aplicável'
        blocks.append(
            f"[Fonte: {document.metadata.get('source')} | página: {page_label} | "
            f"chunk: {document.metadata.get('chunk_id')}]\n{document.page_content}"
        )
    return '\n\n---\n\n'.join(blocks)

documents = [document for document, _ in results]
context = format_context(documents)
print(context[:1500])

## 3. Prompt do RAGnaldo

O humor fica subordinado à evidência. O prompt proíbe conhecimento externo para respostas factuais e exige uma recusa clara quando o contexto for insuficiente.

In [ ]:
rag_prompt = ChatPromptTemplate.from_messages([
    (
        'system',
        '''Você é o RAGnaldo, um guia independente e bem-humorado sobre o ONE AI for Tech.
Responda fatos usando exclusivamente o CONTEXTO fornecido.
Não invente datas, regras, fontes, páginas ou detalhes técnicos.
Se o contexto não for suficiente, diga claramente que não encontrou a informação.
Use humor em no máximo uma frase curta e nunca dentro de uma citação.
Ao final, liste as fontes realmente usadas.

CONTEXTO:
{context}'''
    ),
    ('human', '{question}'),
])

prompt_value = rag_prompt.invoke({'context': context, 'question': question})
print(prompt_value.to_string()[:2500])

## 4. Próxima decisão: modelo gerador

A recuperação está independente do provedor. Depois do benchmark de retrieval, conectaremos uma LLM por adaptador LangChain e formaremos a cadeia `retriever → contexto → prompt → modelo → parser`. Essa decisão permanece aberta para não acoplar o projeto a uma API antes de avaliar custo, qualidade e disponibilidade.

In [ ]:
# Estrutura prevista após a escolha do provedor:
# from langchain_core.output_parsers import StrOutputParser
# chain = rag_prompt | llm | StrOutputParser()
# answer = chain.invoke({'context': context, 'question': question})